In [1]:
## Model

'''
So what I have understood is that I need to take the hidden representations of the LSTM and then use them to get the weights

Input > LSTM > Hidden Representation > Softmax Layer > Weights > Calculate Sharpe (Loss) > Train the Model

# So for simplicity, I will choose 5 stocks to invest in instead of using a 50 stocks and choosing them
the stocks will be namely > TATASTEEL.NS, SUNPHARMA.NS, RELIANCE.NS, INFY.NS, TATACONSUM.NS

'''

'\nSo what I have understood is that I need to take the hidden representations of the LSTM and then use them to get the weights\n\nInput > LSTM > Hidden Representation > Softmax Layer > Weights > Calculate Sharpe (Loss) > Train the Model\n\n# So for simplicity, I will choose 5 stocks to invest in instead of using a 50 stocks and choosing them\nthe stocks will be namely > TATASTEEL.NS, SUNPHARMA.NS, RELIANCE.NS, INFY.NS, TATACONSUM.NS\n\n'

In [2]:
import pandas as pd
import numpy as np
import yaml
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
path = os.environ['PROJECT_FOLDER']

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
dataset = pd.read_csv(os.path.join(path,'data','final','Dataset.csv'))
stocks = pd.read_csv(os.path.join(path,'data','final','Stocks.csv'))

dataset.set_index('Date',inplace=True)
stocks.set_index('Date',inplace=True)

stocks = stocks[['TATASTEEL.NS_7DReturn','SUNPHARMA.NS_7DReturn','RELIANCE.NS_7DReturn','INFY.NS_7DReturn','TATACONSUM.NS_7DReturn']]

In [6]:
class MyDataset(Dataset):
    def __init__(self,X,y,lookback=7):
        self.X = X
        self.y = y
        self.lookback = lookback

        self.X = self.X.reset_index(drop=True)
        self.y = self.y.reset_index(drop=True)
        self.X = torch.from_numpy(X.to_numpy(dtype=np.float32)).to(device)
        self.y = torch.from_numpy(y.to_numpy(dtype=np.float32)).to(device)
    
    def __len__(self):
        return (self.X.shape[0] - self.lookback + 1)
    
    def __getitem__(self,idx):
        X = self.X[idx: idx + self.lookback]
        y = self.y[idx: idx + self.lookback]

        return X,y


In [7]:
dataset = MyDataset(X=dataset,y=stocks)

In [8]:
dataloader = DataLoader(
    dataset=dataset,
    batch_size=24,
    shuffle=False
)

In [9]:
dataset[0]

(tensor([[-3.9809e-01, -1.9981e-01, -3.8730e-01, -1.0931e+00, -3.6665e-01,
           1.6640e-02, -3.5670e-01, -1.1802e-02, -1.4431e-01, -1.5083e-01,
          -2.4695e-01, -3.5750e-01,  2.6634e+00,  3.4853e+00,  3.1018e+00,
           1.2626e+00, -1.2650e-01,  2.5680e-02, -1.4988e-01,  6.2666e-02,
          -3.5504e-01,  8.4214e-04, -4.0474e-01,  3.8145e-02,  6.6535e-02,
           5.4971e-02,  1.5564e-02,  8.6971e-02, -2.2921e-01,  8.7467e-03,
          -2.7002e-01,  3.8324e-02, -4.5003e-01,  5.3715e-02, -4.3898e-01,
           1.3949e-01, -6.4106e-01, -4.6652e-02, -6.5189e-01, -1.4199e-01,
           4.6402e-01,  9.5893e-01,  5.1101e-01,  1.6210e+00,  1.6011e+00,
          -2.3716e+00,  1.8251e+00, -1.4802e+00,  2.6336e-02,  3.1569e-01,
          -5.2299e-02,  8.1708e-01,  6.0998e-01, -2.1039e-01,  3.8362e-01,
          -3.1268e-01,  4.7234e-01, -1.0523e+00,  4.9266e-01, -1.1235e+00,
           7.1537e-01, -5.5139e-01,  4.6290e-01, -6.3899e-01,  7.9505e-03,
          -7.2328e-01, -1

In [10]:
def loss_function(weights,returns):
    portfolio_returns = (weights * returns).sum(dim=1)

    mean_return = portfolio_returns.mean()
    volatility = portfolio_returns.std()

    sharpe = mean_return / (volatility + 1e-8)

    return -sharpe

In [15]:
class lstm_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=76,
            hidden_size=35,
            num_layers=1,    # This adds sequential layers where each layer takes input of prev layer
            # dropout=0.2,
            bias=False,
            batch_first=True
        )
        self.phead = nn.Linear(                     # This is the prediction head
            in_features=35,
            out_features=5
        )
    
    def forward(self,x):
        
        output,(h_n,c_n)=self.lstm(x)

        last_hidden = h_n[-1]
        weights = self.phead(last_hidden)

        softmax = nn.Softmax(dim=1)
        weights = softmax(weights)
        print(weights)
        weights = weights[-1]


        return weights

In [16]:
model = lstm_model().to(device=device)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [13]:
num_epochs=5

In [17]:
for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for X, y in dataloader:
        
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        prediction = model(X)

        loss = loss_function(prediction,y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{running_loss/len(dataloader):.4f}"
    )

tensor([[0.1709, 0.2247, 0.2146, 0.1787, 0.2112],
        [0.1696, 0.2257, 0.2031, 0.2114, 0.1902],
        [0.1643, 0.2145, 0.2176, 0.1922, 0.2114],
        [0.1833, 0.2145, 0.2183, 0.1973, 0.1866],
        [0.1832, 0.2389, 0.2041, 0.1919, 0.1819],
        [0.1903, 0.2458, 0.2054, 0.1870, 0.1715],
        [0.1864, 0.2464, 0.2101, 0.1916, 0.1655],
        [0.1619, 0.2220, 0.2275, 0.2008, 0.1878],
        [0.1630, 0.2596, 0.2248, 0.1839, 0.1687],
        [0.1654, 0.2633, 0.2120, 0.1901, 0.1692],
        [0.1898, 0.2483, 0.2014, 0.1887, 0.1719],
        [0.1726, 0.2437, 0.2129, 0.2036, 0.1672],
        [0.1899, 0.2419, 0.2010, 0.2007, 0.1664],
        [0.1759, 0.2417, 0.2062, 0.1900, 0.1861],
        [0.1793, 0.2034, 0.2202, 0.1936, 0.2036],
        [0.1734, 0.2261, 0.2277, 0.1716, 0.2013],
        [0.1792, 0.2293, 0.2086, 0.1903, 0.1926],
        [0.1732, 0.2005, 0.2108, 0.2137, 0.2018],
        [0.1699, 0.2249, 0.2218, 0.1951, 0.1883],
        [0.1759, 0.2251, 0.2259, 0.2013, 0.1718],


In [141]:
print(dataset[0][0].shape)

X, y = next(iter(dataloader))

print(X.shape)

torch.Size([7, 76])
torch.Size([24, 7, 76])
